In [6]:
import pandas as pd
import numpy as np
from avatars.manager import Manager
from avatars.models import JobKind
import os
import time
import json
import uuid

url = os.environ.get("AVATAR_BASE_API_URL", "https://www.octopize.app/api")
username = os.environ.get("the-chuong.trinh@cea.fr")
password = os.environ.get("ttcAVATARS#123")
# username = os.environ.get("christophe.battail@cea.fr")
# password = os.environ.get("GT5j6ps4n0*!$")

manager = Manager(base_url=url)
manager.authenticate("the-chuong.trinh@cea.fr", "ttcAVATARS#123", should_verify_compatibility=False)

k = 20
#seed = 3

with open("avatars/cluster_final.json", "r") as f:
    cluster_features = json.load(f)

cluster_flat = []
for cluster in cluster_features:
    cluster_flat += cluster
print("Number of features:", len(cluster_flat))
seeds = [0]
ids = [263, 363]
for seed in seeds:
    # os.makedirs(f"avatars/synthetic_blocks_k{k}_{seed}", exist_ok=True)
    errors = []
    for i in ids:
        print(f"---{i}---")
        try:
            data = pd.read_csv(f"avatars/original_blocks/original_block_{i}.csv", index_col =False)
            data_clean = data.drop(columns=["Patient_ID"])
        # Create runner and add table
            job_name = f"Block_{i}_k{k}_{seed}" + str(uuid.uuid4())
            runner = manager.create_runner(job_name, seed = seed)
            table_name = f"block_{i}_k{k}_{seed}" + str(uuid.uuid4())
        
            runner.add_table(table_name, data_clean)
            runner.set_parameters(table_name, k=k)
        
        # # Only run the synthesis job
            avatarization_job = runner.run(jobs_to_run=[JobKind.standard])
        
        # # Now retrieve the unshuffled synthetic data
            synthetic_df = runner.sensitive_unshuffled(table_name)
            synthetic_df.to_csv(f"avatars/synthetic_blocks_k{k}_{seed}/synthetic_block_{i}.csv", index=False)
            if data_clean.shape[1] >= 3000:
                time.sleep(1800)
            else:
                time.sleep(30)
        except Exception as e:
            print(f"Error processing cluster {i}: {e}")
            errors.append((i, str(e)))
            continue

Number of features: 40992
---263---
Creating standard job
Error processing cluster 263: Got error in HTTP request: get /jobs/c43ee4ca-c46d-4327-a19f-0f8363858bf0. Error status 504 - <html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr><center>nginx</center>
</body>
</html>

---363---
Error processing cluster 363: Got error in HTTP request: get /upload_url. Error status 503 - <html>
<head><title>503 Service Temporarily Unavailable</title></head>
<body>
<center><h1>503 Service Temporarily Unavailable</h1></center>
<hr><center>nginx</center>
</body>
</html>



In [4]:
import os
import json
with open("avatars/cluster_final.json", "r") as f:
    cluster_features = json.load(f)
k=20
seed=0
# Folder containing your files
folder = f"avatars/synthetic_blocks_k{k}_{seed}"

# Expected range of i values
expected = set(range(0, len(cluster_features)))  # from 0 to 100 inclusive

# Get all files in folder
files = os.listdir(folder)

# Extract existing i values
existing = set()
for f in files:
    if f.startswith("synthetic_block_") and f.endswith(".csv"):
        try:
            i = int(f.replace("synthetic_block_", "").replace(".csv", ""))
            existing.add(i)
        except ValueError:
            pass  # skip malformed filenames

# Find missing indices
missing = sorted(expected - existing)

print("Missing indices:", missing)


Missing indices: [263, 363]
